# 12F-02 — EPA non-BEV EDA

**Question:** What patterns and quality issues exist in conventional fuel-side results?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
EPA_FILE = REPO_ROOT / "etl/data/raw/epa_testcar/epa_testcar_2026_raw.xlsx"
epa = pd.read_excel(EPA_FILE, sheet_name="Sheet1", engine="openpyxl")


## Keep electric and hydrogen semantics out of this fuel-side view

In [ ]:
fuel_text = epa["Test Fuel Type Description"].fillna("").astype(str)
nonbev = epa[~fuel_text.str.contains("Electricity|Hydrogen", case=False, regex=True)].copy()
nonbev["RND_ADJ_FE"] = pd.to_numeric(nonbev["RND_ADJ_FE"], errors="coerce")
valid_mpg = nonbev["RND_ADJ_FE"].between(0, 500, inclusive="neither")
nonbev["fuel_l_per_100km"] = float("nan")
nonbev.loc[valid_mpg, "fuel_l_per_100km"] = 235.214583 / nonbev.loc[valid_mpg, "RND_ADJ_FE"]
print(f"{len(nonbev):,} source rows; {valid_mpg.sum():,} usable MPG observations")


## Population and missingness

In [ ]:
nonbev["Test Fuel Type Description"].value_counts(dropna=False).head(12)


In [ ]:
review_columns = ["Equivalent Test Weight (lbs.)", "CO2 (g/mi)", "RND_ADJ_FE", "fuel_l_per_100km",
                  "Target Coef A (lbf)", "Target Coef B (lbf/mph)", "Target Coef C (lbf/mph**2)"]
nonbev[review_columns].apply(pd.to_numeric, errors="coerce").describe().T


## Model-year summary

In [ ]:
year_summary = (
    nonbev.assign(co2_g_mi=pd.to_numeric(nonbev["CO2 (g/mi)"], errors="coerce"))
          .groupby("Model Year")
          .agg(source_rows=("Model Year", "size"),
               median_l_100km=("fuel_l_per_100km", "median"),
               median_co2_g_mi=("co2_g_mi", "median"))
)
year_summary


## Does test mass move with fuel consumption?

In [ ]:
import matplotlib.pyplot as plt

plot_data = nonbev.assign(mass_lb=pd.to_numeric(nonbev["Equivalent Test Weight (lbs.)"], errors="coerce"))
plot_data = plot_data.dropna(subset=["mass_lb", "fuel_l_per_100km"])
ax = plot_data.plot.scatter(x="mass_lb", y="fuel_l_per_100km", alpha=0.15, figsize=(7, 4))
ax.set(title="EPA fuel-side observations", xlabel="Equivalent test weight (lb)", ylabel="Fuel consumption (L/100 km)")
plt.show()


## What did we learn?

This is source-row EDA, so repeated tests remain visible. The next modelling steps must use the closed RUN/VDE grain rather than averaging these rows blindly.